In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

DATA_DIR = Path("../data/raw")

customers_path = DATA_DIR / "olist_customers_dataset.csv"
orders_path = DATA_DIR / "olist_orders_dataset.csv"
order_items_path = DATA_DIR / "olist_order_items_dataset.csv"
payments_path = DATA_DIR / "olist_order_payments_dataset.csv"
reviews_path = DATA_DIR / "olist_order_reviews_dataset.csv"
products_path = DATA_DIR / "olist_products_dataset.csv"
sellers_path = DATA_DIR / "olist_sellers_dataset.csv"
geolocation_path = DATA_DIR / "olist_geolocation_dataset.csv"
category_translation_path = DATA_DIR / "product_category_name_translation.csv"
print("ALl good!")


ALl good!


In [2]:
customers = pd.read_csv(customers_path)
orders = pd.read_csv(orders_path)
order_items = pd.read_csv(order_items_path)
payments = pd.read_csv(payments_path)
reviews = pd.read_csv(reviews_path)
products = pd.read_csv(products_path)
sellers = pd.read_csv(sellers_path)
geolocation = pd.read_csv(geolocation_path)
category_translation = pd.read_csv(category_translation_path)

print("All datasets loaded successfully.")

All datasets loaded successfully.


In [3]:
from pathlib import Path

RAW_DIR = Path("../data/raw")

print("Raw folder exists:", RAW_DIR.exists())
print("\nFiles in raw folder:")

for file in RAW_DIR.iterdir():
    print(file.name)

Raw folder exists: True

Files in raw folder:
olist_customers_dataset.csv
olist_geolocation_dataset.csv
olist_orders_dataset.csv
olist_order_items_dataset.csv
olist_order_payments_dataset.csv
olist_order_reviews_dataset.csv
olist_products_dataset.csv
olist_sellers_dataset.csv
product_category_name_translation.csv


In [4]:
from pathlib import Path

print(list(Path.cwd().iterdir()))

[WindowsPath('C:/Users/DELL/Desktop/checkout-incentives-ltv/notebooks/.ipynb_checkpoints'), WindowsPath('C:/Users/DELL/Desktop/checkout-incentives-ltv/notebooks/01_data_audit.ipynb'), WindowsPath('C:/Users/DELL/Desktop/checkout-incentives-ltv/notebooks/02_ltv_modeling.ipynb'), WindowsPath('C:/Users/DELL/Desktop/checkout-incentives-ltv/notebooks/03_checkout_incentive_analysis.ipynb')]


In [5]:
datasets = {
    "Customers": customers,
    "Orders": orders,
    "Order Items": order_items,
    "Payments": payments,
    "Reviews": reviews,
    "Products": products,
    "Sellers": sellers,
    "Geolocation": geolocation,
    "Category Translation": category_translation
}

for name, df in datasets.items():
    print(f"{name:25} {df.shape}")

Customers                 (99441, 5)
Orders                    (99441, 8)
Order Items               (112650, 7)
Payments                  (103886, 5)
Reviews                   (99224, 7)
Products                  (32951, 9)
Sellers                   (3095, 4)
Geolocation               (1000163, 5)
Category Translation      (71, 2)


In [6]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_columns:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")

orders[date_columns].dtypes

order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

In [7]:
customer_orders = orders[
    [
        "order_id",
        "customer_id",
        "order_status",
        "order_purchase_timestamp",
        "order_delivered_customer_date"
    ]
].merge(
    customers[
        [
            "customer_id",
            "customer_unique_id",
            "customer_state"
        ]
    ],
    on="customer_id",
    how="left"
)

customer_orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_delivered_customer_date,customer_unique_id,customer_state
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-10 21:25:13,7c396fd4830fd04220f754e42b4e5bff,SP
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-08-07 15:27:45,af07308b275d755c9edb36a90c618231,BA
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-17 18:06:29,3a653a41f6f9fc3d2a113cf8398680e8,GO
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-12-02 00:28:42,7c142cf63193a1473d2e66489a9ae977,RN
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-16 18:17:02,72632f0f9dd73dfee390c9b22eb56dd6,SP


In [8]:
order_payment_value = (
    payments
    .groupby("order_id", as_index=False)["payment_value"]
    .sum()
    .rename(columns={"payment_value": "order_value"})
)

customer_orders = customer_orders.merge(
    order_payment_value,
    on="order_id",
    how="left"
)

customer_orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_delivered_customer_date,customer_unique_id,customer_state,order_value
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-10 21:25:13,7c396fd4830fd04220f754e42b4e5bff,SP,38.71
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-08-07 15:27:45,af07308b275d755c9edb36a90c618231,BA,141.46
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-17 18:06:29,3a653a41f6f9fc3d2a113cf8398680e8,GO,179.12
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-12-02 00:28:42,7c142cf63193a1473d2e66489a9ae977,RN,72.20
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-16 18:17:02,72632f0f9dd73dfee390c9b22eb56dd6,SP,28.62


In [9]:
customer_features = (
    customer_orders
    .groupby("customer_unique_id")
    .agg(
        total_orders=("order_id", "nunique"),
        total_ltv=("order_value", "sum"),
        first_purchase=("order_purchase_timestamp", "min"),
        last_purchase=("order_purchase_timestamp", "max"),
        customer_state=("customer_state", "first")
    )
    .reset_index()
)

customer_features["customer_lifetime_days"] = (
    customer_features["last_purchase"]
    - customer_features["first_purchase"]
).dt.total_seconds() / (24 * 60 * 60)

customer_features.head()

,customer_unique_id,total_orders,total_ltv,first_purchase,last_purchase,customer_state,customer_lifetime_days
0,0000366f3b9a7992bf8c76cfdf3221e2,1,141.90,2018-05-10 10:56:27,2018-05-10 10:56:27,SP,0.00
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1,27.19,2018-05-07 11:11:27,2018-05-07 11:11:27,SP,0.00
2,0000f46a3911fa3c0805444483337064,1,86.22,2017-03-10 21:05:03,2017-03-10 21:05:03,SC,0.00
3,0000f6ccb0745a6a4b88665a16c9f078,1,43.62,2017-10-12 20:29:41,2017-10-12 20:29:41,PA,0.00
4,0004aac84e0df4da2b147fca70cf8255,1,196.89,2017-11-14 19:45:42,2017-11-14 19:45:42,SP,0.00


In [10]:
customer_features["is_repeat_customer"] = (
    customer_features["total_orders"] > 1
).astype(int)

customer_features["customer_type"] = np.where(
    customer_features["is_repeat_customer"] == 1,
    "Repeat",
    "One-time"
)

customer_features["customer_type"].value_counts()

customer_type
One-time    93099
Repeat       2997
Name: count, dtype: int64

In [11]:
customer_orders = customer_orders.sort_values(
    ["customer_unique_id", "order_purchase_timestamp"]
).reset_index(drop=True)

customer_orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_delivered_customer_date,customer_unique_id,customer_state,order_value
0,e22acc9c116caa3f2b7121bbb380d08e,fadbb3709178fc513abc1b2670aa1ad2,delivered,2018-05-10 10:56:27,2018-05-16 20:48:37,0000366f3b9a7992bf8c76cfdf3221e2,SP,141.90
1,3594e05a005ac4d06a72673270ef9ec9,4cb282e167ae9234755102258dd52ee8,delivered,2018-05-07 11:11:27,2018-05-10 18:02:42,0000b849f77a49e4a4ce2b2a4ca5be3f,SP,27.19
2,b33ec3b699337181488304f362a6b734,9b3932a6253894a02c1df9d19004239f,delivered,2017-03-10 21:05:03,2017-04-05 14:38:47,0000f46a3911fa3c0805444483337064,SC,86.22
3,41272756ecddd9a9ed0180413cc22fb6,914991f0c02ef0843c0e7010c819d642,delivered,2017-10-12 20:29:41,2017-11-01 21:23:05,0000f6ccb0745a6a4b88665a16c9f078,PA,43.62
4,d957021f1127559cd947b62533f484f7,47227568b10f5f58a524a75507e6992c,delivered,2017-11-14 19:45:42,2017-11-27 23:08:56,0004aac84e0df4da2b147fca70cf8255,SP,196.89


In [12]:
customer_orders["purchase_number"] = (
    customer_orders
    .groupby("customer_unique_id")
    .cumcount() + 1
)

customer_orders[
    ["customer_unique_id", "order_id", "order_purchase_timestamp", "purchase_number"]
].head(15)

,customer_unique_id,order_id,order_purchase_timestamp,purchase_number
0,0000366f3b9a7992bf8c76cfdf3221e2,e22acc9c116caa3f2b7121bbb380d08e,2018-05-10 10:56:27,1
1,0000b849f77a49e4a4ce2b2a4ca5be3f,3594e05a005ac4d06a72673270ef9ec9,2018-05-07 11:11:27,1
2,0000f46a3911fa3c0805444483337064,b33ec3b699337181488304f362a6b734,2017-03-10 21:05:03,1
3,0000f6ccb0745a6a4b88665a16c9f078,41272756ecddd9a9ed0180413cc22fb6,2017-10-12 20:29:41,1
4,0004aac84e0df4da2b147fca70cf8255,d957021f1127559cd947b62533f484f7,2017-11-14 19:45:42,1
5,0004bd2a26a76fe21f786e4fbd80607f,3e470077b690ea3e3d501cffb5e0c499,2018-04-05 19:33:16,1
6,00050ab1314c0e55a6ca13cf7181fecf,d0028facea13f508e880202d7097a5a1,2018-04-20 12:57:23,1
7,00053a61a98854899e70ed204dd4bafe,44e608f2db00c74a1fe329de44416a4e,2018-02-28 11:15:41,1
8,0005e1862207bf6ccc02e4228effd9a0,ae76bef74b97bcb0b3e355e60d9a6f9c,2017-03-04 23:32:12,1
9,0005ef4cd20d2893f0d9fbd94d3c0d97,01b330808c5819a6a3cb79b72f0b8288,2018-03-12 15:22:12,1


In [13]:
first_purchases = (
    customer_orders[customer_orders["purchase_number"] == 1]
    [["customer_unique_id", "order_id", "order_value", "order_purchase_timestamp"]]
    .rename(columns={
        "order_id": "first_order_id",
        "order_value": "first_order_value",
        "order_purchase_timestamp": "first_purchase"
    })
)

second_purchases = (
    customer_orders[customer_orders["purchase_number"] == 2]
    [["customer_unique_id", "order_id", "order_value", "order_purchase_timestamp"]]
    .rename(columns={
        "order_id": "second_order_id",
        "order_value": "second_order_value",
        "order_purchase_timestamp": "second_purchase"
    })
)

repeat_timing = first_purchases.merge(
    second_purchases,
    on="customer_unique_id",
    how="inner"
)

repeat_timing.head()

,customer_unique_id,first_order_id,first_order_value,first_purchase,second_order_id,second_order_value,second_purchase
0,00172711b30d52eea8b313a7f2cced02,bb874c45df1a3c97842d52f31efee99a,122.07,2018-07-28 00:23:49,c306eca42d32507b970739b5b6a5a33a,122.07,2018-08-13 09:14:07
1,004288347e5e88a27ded2bb23747066c,a61d617fbe5bd006e40d3a0988fc844b,251.09,2017-07-27 14:13:03,08204559bebd39e09ee52dcb56d8faa2,103.28,2018-01-14 07:36:54
2,004b45ec5c64187465168251cd1c9c2f,90ae229a4addcfead792e2564554f09c,97.87,2017-09-01 12:11:23,9392c5e72885ad5aba87e6223ca9838d,49.85,2018-05-26 19:42:48
3,0058f300f57d7b93c477a131a59b36c3,2cfc79d9582e9135c0a9b61fa60e6b21,79.56,2018-02-19 17:11:34,81a93b2fa39e104b865b2bc471c16008,96.02,2018-03-22 18:09:41
4,00a39521eb40f7012db50455bf083460,7d32c87acba91ed87ebd98310fe1c54d,96.47,2018-05-23 20:14:21,cea3e6c11eb60acb9d8d4d51694832f8,26.78,2018-06-03 10:12:57


In [14]:
repeat_timing["days_to_second_purchase"] = (
    repeat_timing["second_purchase"]
    - repeat_timing["first_purchase"]
).dt.total_seconds() / (24 * 60 * 60)

repeat_timing["days_to_second_purchase"].describe()

count   2,997.00
mean       80.35
std       110.19
min         0.00
25%         0.00
50%        27.92
75%       123.11
max       608.98
Name: days_to_second_purchase, dtype: float64

In [15]:
def classify_repeat_window(days):
    if days < 0:
        return "Invalid"
    elif days <= 7:
        return "0–7 days"
    elif days <= 30:
        return "8–30 days"
    elif days <= 60:
        return "31–60 days"
    elif days <= 90:
        return "61–90 days"
    elif days <= 180:
        return "91–180 days"
    else:
        return "180+ days"

repeat_timing["repeat_window"] = (
    repeat_timing["days_to_second_purchase"]
    .apply(classify_repeat_window)
)

repeat_timing["repeat_window"].value_counts().sort_index()

repeat_window
0–7 days       1097
180+ days       515
31–60 days      320
61–90 days      208
8–30 days       433
91–180 days     424
Name: count, dtype: int64

In [16]:
repeat_window_analysis = (
    repeat_timing
    .groupby("repeat_window")
    .agg(
        repeat_customers=("customer_unique_id", "nunique"),
        avg_days=("days_to_second_purchase", "mean"),
        median_days=("days_to_second_purchase", "median"),
        avg_second_order_value=("second_order_value", "mean")
    )
    .reset_index()
)

repeat_window_analysis

,repeat_window,repeat_customers,avg_days,median_days,avg_second_order_value
0,0–7 days,1097,0.62,0.00,147.79
1,180+ days,515,286.56,266.67,141.84
2,31–60 days,320,44.11,43.94,156.45
3,61–90 days,208,73.73,73.53,151.36
4,8–30 days,433,17.21,16.35,162.08
5,91–180 days,424,131.26,127.17,144.10


In [17]:
genuine_repeat_timing = repeat_timing[
    repeat_timing["days_to_second_purchase"] > 0
].copy()

print("Total repeat customers:", len(repeat_timing))
print("Same-day repeat customers:", (
    repeat_timing["days_to_second_purchase"] == 0
).sum())
print("Genuine next-day-or-later repeats:", len(genuine_repeat_timing))

Total repeat customers: 2997
Same-day repeat customers: 276
Genuine next-day-or-later repeats: 2721


In [18]:
print("=========================")
genuine_repeat_timing["repeat_window"] = (
    genuine_repeat_timing["days_to_second_purchase"]
    .apply(classify_repeat_window)
)

genuine_repeat_window_analysis = (
    genuine_repeat_timing
    .groupby("repeat_window")
    .agg(
        repeat_customers=("customer_unique_id", "nunique"),
        avg_days=("days_to_second_purchase", "mean"),
        median_days=("days_to_second_purchase", "median"),
        avg_second_order_value=("second_order_value", "mean")
    )
    .reset_index()
)

genuine_repeat_window_analysis

,repeat_window,repeat_customers,avg_days,median_days,avg_second_order_value
0,0–7 days,821,0.82,0.00,152.31
1,180+ days,515,286.56,266.67,141.84
2,31–60 days,320,44.11,43.94,156.45
3,61–90 days,208,73.73,73.53,151.36
4,8–30 days,433,17.21,16.35,162.08
5,91–180 days,424,131.26,127.17,144.10


In [19]:
window_order = [
    "0–7 days",
    "8–30 days",
    "31–60 days",
    "61–90 days",
    "91–180 days",
    "180+ days"
]

genuine_repeat_window_analysis["repeat_window"] = pd.Categorical(
    genuine_repeat_window_analysis["repeat_window"],
    categories=window_order,
    ordered=True
)

genuine_repeat_window_analysis = (
    genuine_repeat_window_analysis
    .sort_values("repeat_window")
    .reset_index(drop=True)
)

genuine_repeat_window_analysis

,repeat_window,repeat_customers,avg_days,median_days,avg_second_order_value
0,0–7 days,821,0.82,0.00,152.31
1,8–30 days,433,17.21,16.35,162.08
2,31–60 days,320,44.11,43.94,156.45
3,61–90 days,208,73.73,73.53,151.36
4,91–180 days,424,131.26,127.17,144.10
5,180+ days,515,286.56,266.67,141.84


In [20]:
# Create more precise repeat-purchase windows.
# We separate "Same day" orders from genuine 1–7 day repeats because
# same-day orders may represent a customer splitting one shopping session
# into multiple orders rather than true repeat-purchase behavior.

def classify_precise_repeat_window(days):
    if days == 0:
        return "Same day"
    elif days <= 7:
        return "1–7 days"
    elif days <= 30:
        return "8–30 days"
    elif days <= 60:
        return "31–60 days"
    elif days <= 90:
        return "61–90 days"
    elif days <= 180:
        return "91–180 days"
    else:
        return "180+ days"


# Apply the classification to every customer's time to second purchase.
repeat_timing["precise_repeat_window"] = (
    repeat_timing["days_to_second_purchase"]
    .apply(classify_precise_repeat_window)
)

In [21]:
# Summarize repeat-purchase behavior within each timing window.
# We calculate customer count, average/median time to repeat,
# and average value of the second order to understand both
# timing and the economic value of repeat purchases.

precise_window_analysis = (
    repeat_timing
    .groupby("precise_repeat_window")
    .agg(
        repeat_customers=("customer_unique_id", "nunique"),
        avg_days=("days_to_second_purchase", "mean"),
        median_days=("days_to_second_purchase", "median"),
        avg_second_order_value=("second_order_value", "mean")
    )
    .reset_index()
)


# Define the correct business order for the timing windows.
# Pandas would otherwise sort these alphabetically.

window_order = [
    "Same day",
    "1–7 days",
    "8–30 days",
    "31–60 days",
    "61–90 days",
    "91–180 days",
    "180+ days"
]


# Convert the window column to an ordered categorical variable
# so the final table follows the actual customer journey.

precise_window_analysis["precise_repeat_window"] = pd.Categorical(
    precise_window_analysis["precise_repeat_window"],
    categories=window_order,
    ordered=True
)


# Sort the final analysis according to the defined business order.

precise_window_analysis = (
    precise_window_analysis
    .sort_values("precise_repeat_window")
    .reset_index(drop=True)
)


# Display the final repeat-purchase timing analysis.
precise_window_analysis

,precise_repeat_window,repeat_customers,avg_days,median_days,avg_second_order_value
0,Same day,276,0.00,0.00,134.36
1,1–7 days,821,0.82,0.00,152.31
2,8–30 days,433,17.21,16.35,162.08
3,31–60 days,320,44.11,43.94,156.45
4,61–90 days,208,73.73,73.53,151.36
5,91–180 days,424,131.26,127.17,144.10
6,180+ days,515,286.56,266.67,141.84


In [24]:
# Create a fixed list of the current global variables first.
# This prevents the "dictionary changed size during iteration" error.

global_variables = list(globals().items())

# Search through the variables for a DataFrame containing
# the customer-level LTV columns we need.

for name, obj in global_variables:
    if isinstance(obj, pd.DataFrame):
        
        required_columns = {
            "customer_unique_id",
            "total_orders",
            "total_ltv"
        }
        
        # Check whether this DataFrame contains all required columns.
        if required_columns.issubset(obj.columns):
            print(f"Possible customer LTV dataframe: {name}")
            print(f"Shape: {obj.shape}")
            print(f"Columns: {list(obj.columns)}")

Possible customer LTV dataframe: customer_features
Shape: (96096, 9)
Columns: ['customer_unique_id', 'total_orders', 'total_ltv', 'first_purchase', 'last_purchase', 'customer_state', 'customer_lifetime_days', 'is_repeat_customer', 'customer_type']
Possible customer LTV dataframe: _9
Shape: (5, 7)
Columns: ['customer_unique_id', 'total_orders', 'total_ltv', 'first_purchase', 'last_purchase', 'customer_state', 'customer_lifetime_days']


In [25]:
# Select customers who have made exactly one order.
# These customers are the main target population for the incentive analysis
# because they have purchased once but have not yet demonstrated repeat behavior.

one_time_customers = customer_features[
    customer_features["total_orders"] == 1
].copy()

# Check how many one-time customers we have.
print("One-time customers:", len(one_time_customers))

# Display the first few records to verify the resulting dataset.
one_time_customers.head()

One-time customers: 93099


,customer_unique_id,total_orders,total_ltv,first_purchase,last_purchase,customer_state,customer_lifetime_days,is_repeat_customer,customer_type
0,0000366f3b9a7992bf8c76cfdf3221e2,1,141.90,2018-05-10 10:56:27,2018-05-10 10:56:27,SP,0.00,0,One-time
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1,27.19,2018-05-07 11:11:27,2018-05-07 11:11:27,SP,0.00,0,One-time
2,0000f46a3911fa3c0805444483337064,1,86.22,2017-03-10 21:05:03,2017-03-10 21:05:03,SC,0.00,0,One-time
3,0000f6ccb0745a6a4b88665a16c9f078,1,43.62,2017-10-12 20:29:41,2017-10-12 20:29:41,PA,0.00,0,One-time
4,0004aac84e0df4da2b147fca70cf8255,1,196.89,2017-11-14 19:45:42,2017-11-14 19:45:42,SP,0.00,0,One-time


In [26]:
# Examine the distribution of LTV among one-time customers.
# Since these customers have exactly one order, their total LTV
# represents the monetary value of their single purchase.

one_time_customers["total_ltv"].describe()

count   93,099.00
mean       161.82
std        223.95
min          0.00
25%         62.16
50%        105.70
75%        177.37
max     13,664.08
Name: total_ltv, dtype: float64

In [27]:
# Calculate the median LTV across the complete customer population.
# We use this as the value threshold for customer segmentation.

median_ltv = customer_features["total_ltv"].median()

print(f"Median customer LTV: ${median_ltv:.2f}")


# Classify one-time customers according to whether their LTV
# is above or at/below the overall customer median.

one_time_customers["value_segment"] = np.where(
    one_time_customers["total_ltv"] > median_ltv,
    "High-value",
    "Low-value"
)


# Count customers in each value segment.
one_time_customers["value_segment"].value_counts()

Median customer LTV: $108.00


value_segment
Low-value     47700
High-value    45399
Name: count, dtype: int64

In [28]:
# Compare the number of customers, average LTV, median LTV,
# and total revenue contribution of high- and low-value one-time customers.
#
# This tells us whether the two groups are economically different enough
# to justify different incentive strategies.

one_time_value_summary = (
    one_time_customers
    .groupby("value_segment")
    .agg(
        customers=("customer_unique_id", "nunique"),
        average_ltv=("total_ltv", "mean"),
        median_ltv=("total_ltv", "median"),
        total_revenue=("total_ltv", "sum")
    )
    .reset_index()
)

# Calculate each segment's percentage of the one-time customer population.
one_time_value_summary["customer_share_pct"] = (
    one_time_value_summary["customers"]
    / len(one_time_customers)
    * 100
)

# Calculate each segment's percentage of one-time customer revenue.
one_time_value_summary["revenue_share_pct"] = (
    one_time_value_summary["total_revenue"]
    / one_time_customers["total_ltv"].sum()
    * 100
)

# Sort so that the high-value segment appears first.
one_time_value_summary = (
    one_time_value_summary
    .sort_values("value_segment", ascending=False)
    .reset_index(drop=True)
)

one_time_value_summary

,value_segment,customers,average_ltv,median_ltv,total_revenue,customer_share_pct,revenue_share_pct
0,Low-value,47700,63.58,63.00,"3,032,866.88",51.24,20.13
1,High-value,45399,265.03,180.30,"12,031,982.53",48.76,79.87


In [29]:
# Create spending bands for one-time customers based on their
# first (and only) order value.
#
# Because one-time customers have only one order, total_ltv is
# equivalent to their first-order monetary value.

def classify_order_value(value):
    if value < 50:
        return "< $50"
    elif value < 100:
        return "$50–$99"
    elif value < 250:
        return "$100–$249"
    elif value < 500:
        return "$250–$499"
    elif value < 1000:
        return "$500–$999"
    else:
        return "$1,000+"


# Apply the spending-band classification to every one-time customer.
one_time_customers["order_value_band"] = (
    one_time_customers["total_ltv"]
    .apply(classify_order_value)
)


# Define the correct numerical order for the bands.
# Without this, pandas would sort them alphabetically.

value_band_order = [
    "< $50",
    "$50–$99",
    "$100–$249",
    "$250–$499",
    "$500–$999",
    "$1,000+"
]


# Convert the column into an ordered categorical variable.
one_time_customers["order_value_band"] = pd.Categorical(
    one_time_customers["order_value_band"],
    categories=value_band_order,
    ordered=True
)


# Display the number of customers in each spending band.
one_time_customers["order_value_band"].value_counts().sort_index()

order_value_band
< $50        15780
$50–$99      28319
$100–$249    36084
$250–$499     8825
$500–$999     2969
$1,000+       1122
Name: count, dtype: int64

In [30]:
# Summarize each order-value band.
# This lets us see where most one-time customers are concentrated
# and where most of the one-time revenue comes from.

order_value_band_analysis = (
    one_time_customers
    .groupby("order_value_band", observed=True)
    .agg(
        customers=("customer_unique_id", "nunique"),
        average_order_value=("total_ltv", "mean"),
        median_order_value=("total_ltv", "median"),
        total_revenue=("total_ltv", "sum")
    )
    .reset_index()
)


# Calculate the percentage of one-time customers in each band.
order_value_band_analysis["customer_share_pct"] = (
    order_value_band_analysis["customers"]
    / len(one_time_customers)
    * 100
)


# Calculate the percentage of one-time revenue generated by each band.
order_value_band_analysis["revenue_share_pct"] = (
    order_value_band_analysis["total_revenue"]
    / one_time_customers["total_ltv"].sum()
    * 100
)


# Display the final spending-band analysis.
order_value_band_analysis

,order_value_band,customers,average_order_value,median_order_value,total_revenue,customer_share_pct,revenue_share_pct
0,< $50,15780,36.91,37.54,"582,479.46",16.95,3.87
1,$50–$99,28319,73.28,72.15,"2,075,147.46",30.42,13.77
2,$100–$249,36084,155.72,148.15,"5,619,001.72",38.76,37.30
3,$250–$499,8825,337.31,322.95,"2,976,761.52",9.48,19.76
4,$500–$999,2969,681.12,653.21,"2,022,231.78",3.19,13.42
5,"$1,000+",1122,"1,594.68","1,367.62","1,789,227.47",1.21,11.88


In [31]:
# Compare customers who eventually became repeat purchasers
# with customers who remained one-time buyers.
#
# The key question is:
# "Do repeat customers tend to start with a different order value?"
#
# If repeat customers have systematically higher first-order values,
# initial purchase value can help identify customers with greater
# repeat-purchase potential.

# Start with all customer-level features.
customer_behavior_analysis = customer_features.copy()


# Calculate the average and median LTV for each customer type.
# For one-time customers, LTV equals their first-order value.
# For repeat customers, LTV includes all their recorded orders.

customer_type_summary = (
    customer_behavior_analysis
    .groupby("customer_type")
    .agg(
        customers=("customer_unique_id", "nunique"),
        average_ltv=("total_ltv", "mean"),
        median_ltv=("total_ltv", "median"),
        average_orders=("total_orders", "mean"),
        median_orders=("total_orders", "median")
    )
    .reset_index()
)


# Display the comparison.
customer_type_summary

,customer_type,customers,average_ltv,median_ltv,average_orders,median_orders
0,One-time,93099,161.82,105.70,1.00,1.00
1,Repeat,2997,314.99,225.84,2.12,2.00


In [36]:
# ============================================================
# Create order-level dataset for first-order behavior analysis
# ============================================================

# Keep only delivered orders because we want to analyze
# completed purchases and their subsequent repeat behavior.
orders_delivered = orders[
    orders["order_status"] == "delivered"
].copy()


# ------------------------------------------------------------
# Add customer_unique_id
# ------------------------------------------------------------
# The orders table contains customer_id, while the Customers
# table contains customer_unique_id.
#
# We use customer_unique_id because it represents the actual
# customer across potentially multiple customer_id records.

if "customer_unique_id" not in orders_delivered.columns:

    orders_delivered = orders_delivered.merge(
        customers[
            ["customer_id", "customer_unique_id"]
        ],
        on="customer_id",
        how="left"
    )


# ------------------------------------------------------------
# Create order value
# ------------------------------------------------------------
# If order_value already exists, we keep it.
# Otherwise, calculate it from order items as:
#
# order value = product price + freight

if "order_value" not in orders_delivered.columns:

    order_values = (
        order_items
        .groupby("order_id", as_index=False)
        .agg(
            product_value=("price", "sum"),
            freight_value=("freight_value", "sum")
        )
    )

    order_values["order_value"] = (
        order_values["product_value"]
        + order_values["freight_value"]
    )

    orders_delivered = orders_delivered.merge(
        order_values[
            ["order_id", "order_value"]
        ],
        on="order_id",
        how="left"
    )


# ------------------------------------------------------------
# Select the columns required for first-order analysis
# ------------------------------------------------------------

orders_with_values = orders_delivered[
    [
        "order_id",
        "customer_unique_id",
        "order_purchase_timestamp",
        "order_value"
    ]
].copy()


# Convert purchase timestamp to datetime.
# This allows us to correctly determine which order
# occurred first for each customer.

orders_with_values["order_purchase_timestamp"] = pd.to_datetime(
    orders_with_values["order_purchase_timestamp"]
)


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("Shape:", orders_with_values.shape)

print("\nColumns:")
print(orders_with_values.columns.tolist())

print("\nMissing values:")
print(orders_with_values.isna().sum())

orders_with_values.head()

Shape: (96478, 4)

Columns:
['order_id', 'customer_unique_id', 'order_purchase_timestamp', 'order_value']

Missing values:
order_id                    0
customer_unique_id          0
order_purchase_timestamp    0
order_value                 0
dtype: int64


,order_id,customer_unique_id,order_purchase_timestamp,order_value
0,e481f51cbdc54678b7cc49136f2d6af7,7c396fd4830fd04220f754e42b4e5bff,2017-10-02 10:56:33,38.71
1,53cdb2fc8bc7dce0b6741e2150273451,af07308b275d755c9edb36a90c618231,2018-07-24 20:41:37,141.46
2,47770eb9100c2d0c44946d9cf07ec65d,3a653a41f6f9fc3d2a113cf8398680e8,2018-08-08 08:38:49,179.12
3,949d5b44dbf5de918fe9c16f97b45f8a,7c142cf63193a1473d2e66489a9ae977,2017-11-18 19:28:06,72.20
4,ad21c59c0840e6cb83a9ceb5573f8159,72632f0f9dd73dfee390c9b22eb56dd6,2018-02-13 21:18:39,28.62


In [37]:
# ============================================================
# Identify each customer's first order and compare
# first-order value by eventual customer type
# ============================================================

# Sort every customer's orders chronologically.
# The earliest purchase will therefore become purchase number 1.
orders_with_values = orders_with_values.sort_values(
    ["customer_unique_id", "order_purchase_timestamp"]
).copy()


# Assign a purchase number to every order for each customer.
#
# 1 = first purchase
# 2 = second purchase
# 3 = third purchase
# etc.
orders_with_values["purchase_number"] = (
    orders_with_values
    .groupby("customer_unique_id")
    .cumcount()
    + 1
)


# Keep only the first order of every customer.
# This gives us the initial purchase made by each customer.
first_orders = orders_with_values[
    orders_with_values["purchase_number"] == 1
].copy()


# Add customer_type from the customer_features dataframe
# created earlier in Notebook 02.
#
# customer_type tells us whether the customer eventually became:
# - One-time
# - Repeat
first_orders = first_orders.merge(
    customer_features[
        ["customer_unique_id", "customer_type"]
    ],
    on="customer_unique_id",
    how="left"
)


# Compare first-order value between eventual One-time
# and eventual Repeat customers.
first_order_behavior = (
    first_orders
    .groupby("customer_type")
    .agg(
        customers=("customer_unique_id", "nunique"),
        average_first_order_value=("order_value", "mean"),
        median_first_order_value=("order_value", "median"),
        total_first_order_revenue=("order_value", "sum")
    )
    .reset_index()
)


# Calculate what percentage of customers belongs to
# each customer type.
first_order_behavior["customer_share_pct"] = (
    first_order_behavior["customers"]
    / first_order_behavior["customers"].sum()
    * 100
)


# Calculate what percentage of first-order revenue
# comes from each customer type.
first_order_behavior["revenue_share_pct"] = (
    first_order_behavior["total_first_order_revenue"]
    / first_order_behavior["total_first_order_revenue"].sum()
    * 100
)


# Display the final comparison table.
first_order_behavior

,customer_type,customers,average_first_order_value,median_first_order_value,total_first_order_revenue,customer_share_pct,revenue_share_pct
0,One-time,90379,160.65,105.40,"14,519,216.15",96.81,97.04
1,Repeat,2979,148.49,101.14,"442,339.44",3.19,2.96


In [38]:
# ============================================================
# First-order value bands and eventual repeat rate
# ============================================================

# Create value bands for the customer's first purchase.
# These bands allow us to see whether customers with larger
# initial purchases are actually more likely to repeat.

first_orders["first_order_value_band"] = pd.cut(
    first_orders["order_value"],
    bins=[-float("inf"), 50, 100, 250, 500, 1000, float("inf")],
    labels=[
        "< $50",
        "$50–$99",
        "$100–$249",
        "$250–$499",
        "$500–$999",
        "$1,000+"
    ]
)


# Calculate the number of customers and repeat customers
# within each first-order value band.

first_order_value_analysis = (
    first_orders
    .groupby("first_order_value_band", observed=False)
    .agg(
        customers=("customer_unique_id", "nunique"),
        repeat_customers=("customer_type", lambda x: (x == "Repeat").sum()),
        average_first_order_value=("order_value", "mean"),
        median_first_order_value=("order_value", "median")
    )
    .reset_index()
)


# Calculate the percentage of customers in each band
# who eventually became repeat purchasers.
#
# This is the key metric for evaluating whether first-order
# monetary value can be used as an incentive-targeting signal.

first_order_value_analysis["repeat_rate_pct"] = (
    first_order_value_analysis["repeat_customers"]
    / first_order_value_analysis["customers"]
    * 100
)


# Calculate the percentage of the overall customer base
# represented by each first-order value band.

first_order_value_analysis["customer_share_pct"] = (
    first_order_value_analysis["customers"]
    / first_order_value_analysis["customers"].sum()
    * 100
)


# Display the analysis.
first_order_value_analysis

,first_order_value_band,customers,repeat_customers,average_first_order_value,median_first_order_value,repeat_rate_pct,customer_share_pct
0,< $50,15963,544,36.95,37.60,3.41,17.10
1,$50–$99,28411,931,73.30,72.15,3.28,30.43
2,$100–$249,36240,1150,155.70,148.15,3.17,38.82
3,$250–$499,8763,253,337.36,323.39,2.89,9.39
4,$500–$999,2898,79,680.17,649.84,2.73,3.10
5,"$1,000+",1083,22,"1,587.23","1,351.51",2.03,1.16


In [41]:
# ============================================================
# Repeat behavior by customer state
# ============================================================

# `first_orders` contains one row per customer, representing
# their first purchase.
#
# `customer_features` already contains the customer's state
# and eventual customer type, so we use it directly instead
# of trying to reconstruct customer_state from the raw tables.


# Add customer state to each customer's first-order record.

first_orders_state = first_orders.merge(
    customer_features[
        [
            "customer_unique_id",
            "customer_state"
        ]
    ],
    on="customer_unique_id",
    how="left"
)


# ------------------------------------------------------------
# Validate the merge
# ------------------------------------------------------------

print("Missing customer states:",
      first_orders_state["customer_state"].isna().sum())


# ------------------------------------------------------------
# Calculate repeat behavior by state
# ------------------------------------------------------------

state_repeat_analysis = (
    first_orders_state
    .groupby("customer_state")
    .agg(
        # Total customers whose first purchase occurred
        # in this state.
        customers=("customer_unique_id", "nunique"),

        # Number of those customers who eventually
        # became repeat purchasers.
        repeat_customers=(
            "customer_type",
            lambda x: (x == "Repeat").sum()
        ),

        # Average value of the customer's first purchase.
        average_first_order_value=("order_value", "mean"),

        # Median first-purchase value.
        median_first_order_value=("order_value", "median")
    )
    .reset_index()
)


# Calculate the percentage of customers in each state
# who eventually became repeat purchasers.

state_repeat_analysis["repeat_rate_pct"] = (
    state_repeat_analysis["repeat_customers"]
    / state_repeat_analysis["customers"]
    * 100
)


# Sort states from highest to lowest repeat rate.

state_repeat_analysis = (
    state_repeat_analysis
    .sort_values(
        "repeat_rate_pct",
        ascending=False
    )
    .reset_index(drop=True)
)


# Display the final analysis.

state_repeat_analysis

Missing customer states: 0


,customer_state,customers,repeat_customers,average_first_order_value,median_first_order_value,repeat_rate_pct
0,AC,76,4,245.85,160.15,5.26
1,RO,230,10,239.72,157.88,4.35
2,RJ,11912,420,166.71,112.56,3.53
3,GO,1894,65,171.73,113.44,3.43
4,MT,855,29,206.36,125.91,3.39
5,SP,39150,1296,142.70,93.57,3.31
6,RS,5167,168,161.07,108.20,3.25
7,DF,2017,63,167.50,108.81,3.12
8,MG,10997,342,160.71,108.63,3.11
9,PR,4768,148,159.56,104.91,3.10


In [42]:
# ============================================================
# First-order delivery experience vs repeat behavior
# ============================================================

# Start with the first-order dataset.
# We need the delivery timestamp from the order-level data.

first_order_delivery = first_orders[
    [
        "order_id",
        "customer_unique_id",
        "order_purchase_timestamp",
        "order_value",
        "customer_type"
    ]
].copy()


# Add the actual delivery date for each order.
#
# We use the order-level dataset that contains the delivered
# order information and merge using order_id.

first_order_delivery = first_order_delivery.merge(
    orders_delivered[
        [
            "order_id",
            "order_delivered_customer_date"
        ]
    ],
    on="order_id",
    how="left"
)


# Convert both timestamps to datetime.
first_order_delivery["order_purchase_timestamp"] = pd.to_datetime(
    first_order_delivery["order_purchase_timestamp"]
)

first_order_delivery["order_delivered_customer_date"] = pd.to_datetime(
    first_order_delivery["order_delivered_customer_date"]
)


# Calculate the number of days between purchase and delivery.
#
# This represents the customer's observed delivery time
# for their first order.

first_order_delivery["delivery_days"] = (
    first_order_delivery["order_delivered_customer_date"]
    - first_order_delivery["order_purchase_timestamp"]
).dt.total_seconds() / (24 * 60 * 60)


# Remove records where delivery time could not be calculated.
# These are not useful for this particular analysis.

first_order_delivery_valid = first_order_delivery[
    first_order_delivery["delivery_days"].notna()
    & (first_order_delivery["delivery_days"] >= 0)
].copy()


# Compare delivery experience between customers who eventually
# remained one-time buyers and those who eventually repeated.

delivery_repeat_analysis = (
    first_order_delivery_valid
    .groupby("customer_type")
    .agg(
        customers=("customer_unique_id", "nunique"),
        average_delivery_days=("delivery_days", "mean"),
        median_delivery_days=("delivery_days", "median"),
        min_delivery_days=("delivery_days", "min"),
        max_delivery_days=("delivery_days", "max")
    )
    .reset_index()
)


# Display the result.

delivery_repeat_analysis

,customer_type,customers,average_delivery_days,median_delivery_days,min_delivery_days,max_delivery_days
0,One-time,90371,12.57,10.21,0.53,209.63
1,Repeat,2979,12.39,10.37,1.04,88.24


In [45]:
# ============================================================
# First-order review score vs repeat behavior
# ============================================================

# Load the review dataset directly from the raw data folder.
# This avoids depending on whether the dataframe was previously
# created under a particular variable name.

reviews = pd.read_csv(
    "../data/raw/olist_order_reviews_dataset.csv"
)


# Keep only the columns required for this analysis.
#
# review_score tells us how satisfied the customer was.
# order_id allows us to connect the review to the customer's
# first purchase.

review_scores = reviews[
    [
        "order_id",
        "review_score"
    ]
].drop_duplicates("order_id")


# Start with the first-order dataset.
#
# first_orders contains exactly one row per customer,
# representing their first purchase.

first_order_reviews = first_orders[
    [
        "order_id",
        "customer_unique_id",
        "customer_type",
        "order_value"
    ]
].copy()


# Attach the review score belonging to the customer's
# first order.

first_order_reviews = first_order_reviews.merge(
    review_scores,
    on="order_id",
    how="left"
)


# ------------------------------------------------------------
# Compare review scores between one-time and repeat customers
# ------------------------------------------------------------

review_repeat_analysis = (
    first_order_reviews
    .groupby("customer_type")
    .agg(
        # Number of customers in each customer type.
        customers=("customer_unique_id", "nunique"),

        # Number of customers who provided a review score.
        reviewed_customers=("review_score", "count"),

        # Average review score.
        average_review_score=("review_score", "mean"),

        # Median review score.
        median_review_score=("review_score", "median")
    )
    .reset_index()
)


# Calculate the percentage of customers who reviewed
# their first order.

review_repeat_analysis["review_rate_pct"] = (
    review_repeat_analysis["reviewed_customers"]
    / review_repeat_analysis["customers"]
    * 100
)


# Display the result.

review_repeat_analysis

,customer_type,customers,reviewed_customers,average_review_score,median_review_score,review_rate_pct
0,One-time,90379,89785,4.15,5.00,99.34
1,Repeat,2979,2956,4.16,5.00,99.23


In [46]:
# ============================================================
# High-value one-time customers: incentive opportunity
# ============================================================

# A customer is considered:
# 1. One-time      -> exactly one recorded order
# 2. High-value    -> LTV is at or above the median LTV
#
# The median LTV is used as the threshold because the customer
# value distribution is strongly right-skewed. Using the mean
# would be heavily influenced by a small number of very
# high-value customers.


# Calculate the overall median customer LTV.

median_ltv = customer_features["total_ltv"].median()

print("Median customer LTV:", round(median_ltv, 2))


# ------------------------------------------------------------
# Identify high-value one-time customers
# ------------------------------------------------------------

high_value_one_time = customer_features[
    (customer_features["customer_type"] == "One-time")
    & (customer_features["total_ltv"] >= median_ltv)
].copy()


# ------------------------------------------------------------
# Calculate the size and financial importance of this segment
# ------------------------------------------------------------

target_customers = high_value_one_time["customer_unique_id"].nunique()

target_revenue = high_value_one_time["total_ltv"].sum()

target_average_ltv = high_value_one_time["total_ltv"].mean()

target_median_ltv = high_value_one_time["total_ltv"].median()


# Compare the target segment with the complete customer base.

total_customers = customer_features["customer_unique_id"].nunique()

total_revenue = customer_features["total_ltv"].sum()


target_customer_share = (
    target_customers / total_customers * 100
)

target_revenue_share = (
    target_revenue / total_revenue * 100
)


# ------------------------------------------------------------
# Display the opportunity
# ------------------------------------------------------------

target_summary = pd.DataFrame({
    "metric": [
        "Target customers",
        "Target average LTV",
        "Target median LTV",
        "Target revenue",
        "Customer share (%)",
        "Revenue share (%)"
    ],
    "value": [
        target_customers,
        round(target_average_ltv, 2),
        round(target_median_ltv, 2),
        round(target_revenue, 2),
        round(target_customer_share, 2),
        round(target_revenue_share, 2)
    ]
})


target_summary

Median customer LTV: 108.0


,metric,value
0,Target customers,"45,432.00"
1,Target average LTV,264.91
2,Target median LTV,180.21
3,Target revenue,"12,035,546.53"
4,Customer share (%),47.28
5,Revenue share (%),75.18


In [48]:
# ============================================================
# HIGH-VALUE ONE-TIME CUSTOMER: REPEAT-CONVERSION OPPORTUNITY
# ============================================================
#
# Business objective:
# Estimate the size of the high-value one-time customer pool
# and the potential revenue opportunity from converting some
# of these customers into repeat purchasers.
#
# IMPORTANT:
# This is a scenario estimate, NOT a prediction.
# We are using historical repeat behavior only as a benchmark.
# ============================================================


# ------------------------------------------------------------
# 1. Identify high-value one-time customers
# ------------------------------------------------------------

# Use the median LTV as the high-value threshold.
# The median is preferred because LTV is strongly right-skewed.

median_ltv = customer_features["total_ltv"].median()

high_value_one_time = customer_features[
    (customer_features["customer_type"] == "One-time")
    & (customer_features["total_ltv"] >= median_ltv)
].copy()

target_customers = len(high_value_one_time)


# ------------------------------------------------------------
# 2. Calculate the historical repeat rate
# ------------------------------------------------------------

# This represents the proportion of all customers who
# eventually made more than one purchase.

overall_repeat_rate = (
    customer_features["customer_type"].eq("Repeat").mean()
)


# ------------------------------------------------------------
# 3. Estimate potential repeat conversions
# ------------------------------------------------------------

# If the high-value one-time segment converted at the same
# historical repeat rate as the overall customer population,
# this would be the approximate number of repeat conversions.

estimated_conversions = round(
    target_customers * overall_repeat_rate
)


# ------------------------------------------------------------
# 4. Calculate average second-order value
# ------------------------------------------------------------

# Reconstruct the second-order dataset directly from
# `first_orders` and `orders_with_values`.
#
# This avoids depending on an undefined variable such as
# `second_orders`.

repeat_orders = orders_with_values.merge(
    customer_features[
        [
            "customer_unique_id",
            "customer_type"
        ]
    ],
    on="customer_unique_id",
    how="inner"
)

# Keep only customers who actually became repeat purchasers.

repeat_orders = repeat_orders[
    repeat_orders["customer_type"] == "Repeat"
].copy()


# Sort each customer's orders chronologically so that
# purchase number 2 can be identified correctly.

repeat_orders = repeat_orders.sort_values(
    [
        "customer_unique_id",
        "order_purchase_timestamp"
    ]
)


# Number each purchase for every repeat customer.

repeat_orders["purchase_number"] = (
    repeat_orders
    .groupby("customer_unique_id")
    .cumcount()
    + 1
)


# Keep only the second purchase made by each repeat customer.

second_order_values = repeat_orders[
    repeat_orders["purchase_number"] == 2
].copy()


# Calculate the average value of the second purchase.

average_second_order_value = (
    second_order_values["order_value"].mean()
)


# ------------------------------------------------------------
# 5. Estimate potential incremental revenue
# ------------------------------------------------------------

# This represents the revenue that could theoretically be
# generated if the estimated conversions each made a second
# purchase equal to the historical average second-order value.

estimated_incremental_revenue = (
    estimated_conversions
    * average_second_order_value
)


# ------------------------------------------------------------
# 6. Create a clean summary table
# ------------------------------------------------------------

conversion_opportunity = pd.DataFrame({
    "metric": [
        "High-value one-time customers",
        "Historical repeat rate (%)",
        "Estimated repeat conversions",
        "Average second-order value",
        "Estimated incremental revenue"
    ],
    "value": [
        target_customers,
        round(overall_repeat_rate * 100, 2),
        estimated_conversions,
        round(average_second_order_value, 2),
        round(estimated_incremental_revenue, 2)
    ]
})


# Display the result.

conversion_opportunity

,metric,value
0,High-value one-time customers,"45,432.00"
1,Historical repeat rate (%),3.12
2,Estimated repeat conversions,"1,417.00"
3,Average second-order value,146.94
4,Estimated incremental revenue,"208,219.24"


In [49]:
# ============================================================
# INCENTIVE SCENARIO ANALYSIS
# ============================================================
# We now estimate the potential business impact of targeting
# high-value one-time customers with a checkout/repeat-purchase
# incentive.
#
# We will test different repeat-conversion rates and different
# incentive costs per converted customer.
#
# IMPORTANT:
# This is a scenario analysis, NOT a causal estimate.
# We are estimating potential outcomes rather than claiming
# that an incentive will definitely create these conversions.


# ------------------------------------------------------------
# 1. Define the target population
# ------------------------------------------------------------
# These are the high-value one-time customers identified earlier.
# They are the primary target because they have already spent
# substantial amounts but have not yet made a second purchase.

target_customers = 45_432

# Average value of the second order among historical repeat
# customers.
average_second_order_value = 146.94


# ------------------------------------------------------------
# 2. Define hypothetical conversion scenarios
# ------------------------------------------------------------
# We test several possible repeat-conversion rates.
#
# Example:
# A 1% conversion rate means that 1% of the 45,432 target
# customers are assumed to make a second purchase.

conversion_rates = [0.01, 0.03, 0.05, 0.10]


# ------------------------------------------------------------
# 3. Define incentive costs
# ------------------------------------------------------------
# These represent the cost of the incentive given to each
# customer who converts.
#
# We are treating the incentive as a business cost.
# The values are scenario assumptions, not observed costs
# from the Olist dataset.

incentive_costs = [5, 10, 15]


# ------------------------------------------------------------
# 4. Generate all conversion × incentive combinations
# ------------------------------------------------------------
# Each row represents one possible business scenario.

scenarios = []

for conversion_rate in conversion_rates:

    # Estimate how many high-value one-time customers
    # would make a repeat purchase under this scenario.
    estimated_conversions = round(
        target_customers * conversion_rate
    )

    # Estimate the revenue generated by those second purchases.
    estimated_revenue = (
        estimated_conversions
        * average_second_order_value
    )

    for incentive_cost in incentive_costs:

        # Total incentive expenditure is assumed to be paid
        # only for customers who actually convert.
        total_incentive_cost = (
            estimated_conversions
            * incentive_cost
        )

        # Net incremental revenue after deducting incentive cost.
        net_incremental_revenue = (
            estimated_revenue
            - total_incentive_cost
        )

        # ROI measures the return generated relative to
        # the incentive expenditure.
        #
        # ROI = (Net Gain / Incentive Cost) × 100
        roi_pct = (
            net_incremental_revenue
            / total_incentive_cost
            * 100
        )

        scenarios.append({
            "conversion_rate_pct": conversion_rate * 100,
            "estimated_conversions": estimated_conversions,
            "average_second_order_value": average_second_order_value,
            "estimated_revenue": estimated_revenue,
            "incentive_cost_per_conversion": incentive_cost,
            "total_incentive_cost": total_incentive_cost,
            "net_incremental_revenue": net_incremental_revenue,
            "roi_pct": roi_pct
        })


# ------------------------------------------------------------
# 5. Convert the scenarios into a dataframe
# ------------------------------------------------------------
incentive_scenarios = pd.DataFrame(scenarios)


# ------------------------------------------------------------
# 6. Display the scenario analysis
# ------------------------------------------------------------
# Rounding makes the business results easier to read.

incentive_scenarios = incentive_scenarios.round({
    "conversion_rate_pct": 2,
    "average_second_order_value": 2,
    "estimated_revenue": 2,
    "incentive_cost_per_conversion": 2,
    "total_incentive_cost": 2,
    "net_incremental_revenue": 2,
    "roi_pct": 2
})

print("INCENTIVE SCENARIO ANALYSIS")
print("=" * 80)

incentive_scenarios

INCENTIVE SCENARIO ANALYSIS


,conversion_rate_pct,estimated_conversions,average_second_order_value,estimated_revenue,incentive_cost_per_conversion,total_incentive_cost,net_incremental_revenue,roi_pct
0,1.00,454,146.94,"66,710.76",5,2270,"64,440.76","2,838.80"
1,1.00,454,146.94,"66,710.76",10,4540,"62,170.76","1,369.40"
2,1.00,454,146.94,"66,710.76",15,6810,"59,900.76",879.60
3,3.00,1363,146.94,"200,279.22",5,6815,"193,464.22","2,838.80"
4,3.00,1363,146.94,"200,279.22",10,13630,"186,649.22","1,369.40"
5,3.00,1363,146.94,"200,279.22",15,20445,"179,834.22",879.60
6,5.00,2272,146.94,"333,847.68",5,11360,"322,487.68","2,838.80"
7,5.00,2272,146.94,"333,847.68",10,22720,"311,127.68","1,369.40"
8,5.00,2272,146.94,"333,847.68",15,34080,"299,767.68",879.60
9,10.00,4543,146.94,"667,548.42",5,22715,"644,833.42","2,838.80"


In [50]:
# ============================================================
# BREAK-EVEN INCENTIVE ANALYSIS
# ============================================================
#
# Business question:
# What is the maximum incentive cost that could theoretically
# be given to a converted customer before the second purchase
# stops generating positive incremental revenue?
#
# IMPORTANT:
# This is a revenue-level break-even calculation.
# It is NOT a true profit break-even because the Olist dataset
# does not provide product margins, operating costs, or other
# contribution-margin information.
# ============================================================


# ------------------------------------------------------------
# 1. Historical average second-order value
# ------------------------------------------------------------
# This is the average revenue generated by a customer's
# second purchase among historical repeat customers.

average_second_order_value = 146.94


# ------------------------------------------------------------
# 2. Calculate theoretical revenue break-even incentive
# ------------------------------------------------------------
# If the incentive equals the entire second-order revenue,
# the incremental revenue after incentive cost becomes zero.
#
# Therefore:
#
# Break-even incentive = Average second-order value

break_even_incentive = average_second_order_value


# ------------------------------------------------------------
# 3. Evaluate practical incentive levels
# ------------------------------------------------------------
# We test several incentive levels below the theoretical
# break-even point to understand how much revenue remains
# after the incentive cost.

incentive_levels = [5, 10, 15, 25, 50, 75, 100, 125, 146.94]


break_even_analysis = []

for incentive in incentive_levels:

    # Revenue remaining after paying the incentive.
    net_revenue_per_conversion = (
        average_second_order_value - incentive
    )

    # Revenue-level ROI.
    #
    # ROI = Net revenue / Incentive cost × 100
    roi_pct = (
        net_revenue_per_conversion
        / incentive
        * 100
    )

    break_even_analysis.append({
        "incentive_per_conversion": incentive,
        "average_second_order_value": average_second_order_value,
        "net_revenue_per_conversion": net_revenue_per_conversion,
        "roi_pct": roi_pct
    })


# ------------------------------------------------------------
# 4. Create the final table
# ------------------------------------------------------------

break_even_analysis = pd.DataFrame(
    break_even_analysis
)


# Round the financial values for readability.

break_even_analysis = break_even_analysis.round({
    "incentive_per_conversion": 2,
    "average_second_order_value": 2,
    "net_revenue_per_conversion": 2,
    "roi_pct": 2
})


# ------------------------------------------------------------
# 5. Display the analysis
# ------------------------------------------------------------

print("BREAK-EVEN INCENTIVE ANALYSIS")
print("=" * 80)

print(
    f"Theoretical revenue break-even incentive: "
    f"${break_even_incentive:.2f}"
)

break_even_analysis

BREAK-EVEN INCENTIVE ANALYSIS
Theoretical revenue break-even incentive: $146.94


,incentive_per_conversion,average_second_order_value,net_revenue_per_conversion,roi_pct
0,5.00,146.94,141.94,"2,838.80"
1,10.00,146.94,136.94,"1,369.40"
2,15.00,146.94,131.94,879.60
3,25.00,146.94,121.94,487.76
4,50.00,146.94,96.94,193.88
5,75.00,146.94,71.94,95.92
6,100.00,146.94,46.94,46.94
7,125.00,146.94,21.94,17.55
8,146.94,146.94,0.00,0.00
